# xbrl-frontier Phase 2: SEC-BERT encoder baseline on a Kaggle T4 (owner-run, resumable)

`MODE = 'train'` trains on the corpus (resume like the student runs); `MODE = 'predict'` writes encoder predictions for `SET` from a trained model.
**Settings:** GPU T4, Internet on. **Attach:** `xbrl-frontier-src`, `xbrl-frontier-corpus`; to resume or predict, also the previous output dataset.

In [ ]:
MODE = 'train'        # 'train' or 'predict'
RESUME = None         # previous output dataset path when continuing training
MODEL_DIR = None      # predict: path to final_model/ of a finished training run
SET = 'finni'         # predict: 'finni', 'dev' or 'fresh'

In [ ]:
import glob, os, shutil, subprocess, sys
dst = '/kaggle/working/xbrl-frontier'
tarballs = glob.glob('/kaggle/input/**/xbrl-frontier-src.tar.gz', recursive=True)
trees = [os.path.dirname(os.path.dirname(os.path.dirname(p)))
         for p in glob.glob('/kaggle/input/**/docs/config/thresholds.yaml', recursive=True)]
if tarballs:
    assert len(tarballs) == 1, tarballs
    os.makedirs(dst, exist_ok=True)
    subprocess.run(['tar', '-xzf', tarballs[0], '-C', dst], check=True)
else:  # Kaggle auto-extracted the upload
    assert len(trees) == 1, trees
    shutil.copytree(trees[0], dst, dirs_exist_ok=True)
os.chdir(dst)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import yaml
pins = yaml.safe_load(open('docs/config/thresholds.yaml'))['pins']['software']
missing = [k for k in ('transformers', 'accelerate') if not pins.get(k)]
assert not missing, f'pins.software empty for {missing}: owner approval needed first'
print(pins)

In [ ]:
!pip install -q "transformers=={pins['transformers']}" "accelerate=={pins['accelerate']}" "pyyaml==6.0.3" "pyarrow==25.0.1" "huggingface_hub"
import importlib.metadata as md
keys = ('transformers', 'accelerate')
installed = {k: md.version(k) for k in keys}
bad = {k: (v, pins[k]) for k, v in installed.items() if v != pins[k]}
assert not bad, f'installed != pins.software: {bad}; restart the kernel and run all again'
import importlib
imported = {k: importlib.import_module(k).__version__ for k in keys}
bad = {k: (v, pins[k]) for k, v in imported.items() if v != pins[k]}
assert not bad, f'imported != pins.software: {bad}; restart the kernel (Run > Restart) and run all again'
import torch; print(imported, 'torch', torch.__version__, 'CUDA', torch.version.cuda, torch.cuda.get_device_name(0))

In [ ]:
import glob, os
corpus = os.path.dirname(glob.glob('/kaggle/input/**/train.jsonl', recursive=True)[0])
out = '/kaggle/working/run-encoder'
cmd = f'python -m training.encoder_train {MODE} --corpus {corpus} --out {out}'
cmd += f' --resume {RESUME}' if RESUME else ''
cmd += f' --model-dir {MODEL_DIR} --set {SET}' if MODE == 'predict' else ''
print(cmd)
!{cmd}
!ls -la {out}